In [ ]:
# ========== 依赖安装（Colab / 本地均可）==========
# bitsandbytes：量化与加速相关；accelerate：多设备加载；transformers 钉版本避免 API 漂移
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6


In [ ]:
# ========== 导入：本笔记本后续单元格要用到的库 ==========

# 标准库 os：读环境变量（本格主要作通用依赖）
import os
# requests：HTTP 客户端（若后续扩展 API 调用可用）
import requests
# IPython 展示：Markdown / display / 流式更新显示
from IPython.display import Markdown, display, update_display
# OpenAI 官方客户端（云端 Chat Completions；本练习主路径用本地 transformers）
from openai import OpenAI
# Google Colab：挂载 Drive（本练习可选）
from google.colab import drive
# Hugging Face Hub：login 把 token 交给后续 from_pretrained
from huggingface_hub import login
# Colab Secrets：从 userdata 安全读取 HF_TOKEN，避免写进代码
from google.colab import userdata
# transformers：分词器 + 因果语言模型 + 流式打印 + 量化配置类
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
# PyTorch：张量与 CUDA 可用性判断
import torch


In [ ]:
# ========== 第 3 周练习：用开源 Llama 生成印度电商合成客户数据 ==========
# 目标：本地/Colab 加载 Instruct 模型 → 批生成 JSON 客户记录 → 去重 → 存 CSV
# 与本课关系：Hugging Face login、AutoTokenizer / AutoModelForCausalLM、chat template、JSON 清洗

# --- 标准库与第三方 ---
# os：环境相关（本格以 HF token 为主）
import os
# re：从模型原文里用正则抠出 JSON 数组/对象
import re
# json：把字符串解析成 Python list/dict
import json
# torch：dtype / device / cuda 检测
import torch
# pandas：把记录列表变成 DataFrame 并写 CSV
import pandas as pd
# dotenv：本地 .env 加载（Colab 场景下主要用 userdata）
from dotenv import load_dotenv
# Hugging Face：用 token 登录，便于拉取门控模型
from huggingface_hub import login
# transformers：分词器 + 因果 LM
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------------------------------
# 授权：从 Colab Secrets 取 HF_TOKEN 并 login
# -------------------------------
# userdata 来自上格 `from google.colab import userdata`（须先跑导入格）
hf_token = userdata.get('HF_TOKEN')
# add_to_git_credential=True：把凭证写进 git 凭据助手，减少重复登录
login(hf_token, add_to_git_credential=True)

# -------------------------------
# 模型加载：Llama-3.2-1B-Instruct（轻量 Instruct，适合合成数据）
# -------------------------------
# 模型 ID 字符串必须与 Hub 上一致；勿改名
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

# 加载分词器：trust_remote_code 允许仓库自定义代码（按原设置保留）
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# 加载因果语言模型：有 GPU 用 float16 + device_map=auto；否则 float32、稍后 .to("cpu")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

# 无 CUDA 时显式放到 CPU，避免 device_map=None 时设备不一致
if not torch.cuda.is_available():
    model = model.to("cpu")

# -------------------------------
# 文本生成：chat template → generate → 只返回新生成段
# -------------------------------
def generate_text(prompt, max_tokens=400):
    # 单轮 user 消息；Instruct 模型期望 chat 格式
    messages = [{"role": "user", "content": prompt}]

    # apply_chat_template：按模型约定拼 system/user 特殊标记；tokenize=False 先拿字符串
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # 再编码成模型输入张量（input_ids / attention_mask）
    inputs = tokenizer(text, return_tensors="pt")

    # GPU 上把张量搬到与模型同一 device
    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # 采样生成：温度 0.4 兼顾多样性与结构稳定；pad 用 eos 避免警告
    output = model.generate(
        **inputs,
        max_new_tokens=max_tokens,   
        temperature=0.4,            
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # decode 全文后切掉 prompt 前缀，只留模型新写的内容
    return tokenizer.decode(output[0], skip_special_tokens=True)[len(text):].strip()

# -------------------------------
# JSON 提取：模型常夹带说明文字，需鲁棒解析
# -------------------------------
def extract_objects(raw):
    # 1) 优先匹配最外层 JSON 数组 [...]
    array_match = re.search(r"\[.*?\]", raw, re.DOTALL)
    if array_match:
        try:
            parsed = json.loads(array_match.group())
            if isinstance(parsed, list):
                # 只保留 dict 记录，丢掉非对象元素
                return [r for r in parsed if isinstance(r, dict)]
        except json.JSONDecodeError:
            pass

    # 2) 回退：逐个匹配浅层 {...}（不处理深层嵌套）
    matches = re.findall(r"\{[^{}]*\}", raw, re.DOTALL)  
    data = []
    for m in matches:
        try:
            data.append(json.loads(m))
        except json.JSONDecodeError:
            continue
    return data

# -------------------------------
# 批量生成：用城市/品类种子约束多样性
# -------------------------------
# 候选城市 / 品类 / 支付方式：写入 prompt 作提示，减少模型瞎编
CITIES = ["Mumbai", "Delhi", "Pune", "Bangalore", "Chennai", "Hyderabad", "Kolkata", "Jaipur", "Ahmedabad", "Surat"]
CATEGORIES = ["Electronics", "Fashion", "Grocery", "Books", "Sports", "Beauty", "Home Decor", "Toys", "Appliances"]
PAYMENTS = ["UPI", "Credit Card", "Debit Card", "Net Banking", "Cash on Delivery", "Wallet"]

def generate_batch(batch_size=5):
    # 每批随机抽城市与品类种子，逼模型换名字/城市，而不是反复 Rahul Sharma
    import random
    city_hint = random.sample(CITIES, min(batch_size, len(CITIES)))
    category_hint = random.sample(CATEGORIES, min(batch_size, len(CATEGORIES)))

    # prompt 字符串（英文）是可执行契约：要求纯 JSON、字段形状固定——勿翻译
    prompt = f"""Return ONLY a JSON array with exactly {batch_size} records. No explanation. No markdown.

Rules:
- Every "name" must be a DIFFERENT real Indian full name. Do NOT repeat names.
- Use these cities (one per record): {city_hint}
- Use these categories (one per record): {category_hint}
- "age" must be between 21 and 55
- "total_orders" must be between 1 and 20
- "avg_order_value" must be between 500.0 and 9000.0

Output format (use DIFFERENT values, this is just the shape):
[{{"name":"<first> <last>","age":<int>,"city":"<city>","total_orders":<int>,"avg_order_value":<float>,"preferred_category":"<category>","payment_method":"<payment>"}}]"""

    # 调用生成；max_tokens 略大于默认以装下多条 JSON
    raw = generate_text(prompt, max_tokens=500)
    # 去掉 ```json ... ``` 围栏，便于 json.loads
    raw = re.sub(r"```(?:json)?|```", "", raw).strip()
    return extract_objects(raw)


def generate_dataset(n=100):
    # 累积合法记录；seen_names 防止同名重复灌进数据集
    data = []
    seen_names = set()        # 阻断重复姓名
    # 最多尝试 n*6 批，避免模型太重复时死循环
    max_attempts = n * 6
    attempts = 0

    while len(data) < n and attempts < max_attempts:
        # 每批固定要 5 条（模型一次吐一批更稳）
        batch = generate_batch(5)
        attempts += 1

        for record in batch:
            if len(data) >= n:
                break

            # 取姓名；空或已见则跳过
            name = record.get("name", "").strip()

            if not name or name in seen_names:
                continue                          # 跳过克隆名

            # 补上客户 ID：CUST_0001 形式
            record["customer_id"] = f"CUST_{len(data) + 1:04d}"
            data.append(record)
            seen_names.add(name)

        # 进度打印（英文文案保持原样，便于对照输出）
        print(f"Generated: {len(data)} / {n}  (attempt {attempts})")

    if len(data) < n:
        print(f"⚠️  Only got {len(data)} unique records — model too repetitive for n={n}")

    return data[:n]

# -------------------------------
# 主入口：小样本生成 → DataFrame → CSV
# -------------------------------
if __name__ == "__main__":
    # n=10：演示速度优先；正式数据可改大
    data = generate_dataset(10)   # 保持小规模以便快速跑通

    # 列表 of dict → 表格
    df = pd.DataFrame(data)
    # 写出 CSV；index=False 不写行号列
    df.to_csv("synthetic_data.csv", index=False)

    print("\n✅ Done! Sample data:")
    print(df.head())
